# Linear Attention ViT Benchmark: Multi-Architecture Comparison for Particle Collision Images

**GSoC ML4SCI Project** — Research-Grade Experimental Framework

This notebook benchmarks three Vision Transformer architectures on particle collision detector images:
1. **Standard ViT** — quadratic self-attention O(N²·d)
2. **Linear Attention ViT** — Cross-Covariance Attention (XCA) O(N·d²) 
3. **Hybrid CNN+ViT** — CNN backbone + transformer encoder

Architecture diagrams: `../images/`

## Section 1: Configuration

In [ ]:
# ============================================================
# Section 1: Configuration
# All hyperparameters in one place for reproducibility
# ============================================================

import os
import random
import time
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

# ----------------------------------------------------------
# Hyperparameters
# ----------------------------------------------------------

# Model
IMG_SIZE = 64          # Input image resolution
PATCH_SIZE = 8         # Patch size for tokenization
IN_CHANS = 8           # Number of input channels (detector layers)
EMBED_DIM = 256        # Transformer embedding dimension
DEPTH = 10             # Number of transformer blocks
NUM_HEADS = 8          # Number of attention heads
MLP_RATIO = 4.0        # MLP hidden dim ratio
DROPOUT = 0.1          # Dropout probability

# Training
BATCH_SIZE = 32
EPOCHS = 20
LR = 3e-4
WEIGHT_DECAY = 1e-4
TRAIN_FRAC = 0.80      # 80/20 train/val split
SEED = 42
REGRESSION_LAMBDA = 1.0  # Weight for regression (MSE) loss

# Pretraining (MAE-style, not used in benchmark but kept for reference)
PRETRAIN_EPOCHS = 30
MASK_RATIO = 0.50
LR_PRETRAIN = 1e-3

# Data paths
DATA_DIR = Path("../data")
UNLABELED_FILE = DATA_DIR / "Dataset_Specific_Unlabelled.h5"
LABELED_FILE = DATA_DIR / "Dataset_Specific_labelled_full_only_for_2i.h5"

NUM_CLASSES = 2  # quark vs gluon (or whatever the dataset has)


# ----------------------------------------------------------
# Device detection
# ----------------------------------------------------------

def get_device() -> torch.device:
    """Auto-detect the best available compute device."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = get_device()
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


# ----------------------------------------------------------
# Reproducibility — seed everything
# ----------------------------------------------------------

def seed_everything(seed: int = 42) -> None:
    """Set all random seeds for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
print(f"Seeds set to {SEED}")
print(f"PyTorch version: {torch.__version__}")

## Section 2: Dataset Loading

In [ ]:
# ============================================================
# Section 2: Dataset Loading
# Lazy HDF5 loading with synthetic fallback
# ============================================================

import h5py
from torch.utils.data import Dataset, DataLoader, random_split


def inspect_hdf5(filepath: str) -> None:
    """Print the structure and shapes of an HDF5 file."""
    print(f"\n=== HDF5 File: {filepath} ===")
    try:
        with h5py.File(filepath, "r") as f:
            def _print_tree(name, obj):
                indent = "  " * name.count("/")
                if hasattr(obj, "shape"):
                    print(f"{indent}{name}: shape={obj.shape}, dtype={obj.dtype}")
                else:
                    print(f"{indent}{name}/")
            f.visititems(_print_tree)
    except Exception as e:
        print(f"  Could not open file: {e}")


class LazyHDF5Dataset(Dataset):
    """
    Lazy-loading HDF5 dataset for particle collision images.

    Does NOT load the entire dataset into RAM. Opens the HDF5 file
    handle once and reads individual samples in __getitem__.

    Parameters
    ----------
    filepath : str or Path
        Path to the HDF5 file.
    labeled : bool
        If True, returns (image, mass, label) tuples.
        If False, returns (image,) tuples.
    transform : callable, optional
        Optional transform applied to each image tensor.
    """

    def __init__(self, filepath, labeled: bool = True, transform=None):
        self.filepath = str(filepath)
        self.labeled = labeled
        self.transform = transform
        self._file = None  # lazy file handle

        # Open once to get length and key names
        with h5py.File(self.filepath, "r") as f:
            keys = list(f.keys())
            # Try common key names for images
            img_key = None
            for k in ["X", "images", "image", "data", "X_jets"]:
                if k in keys:
                    img_key = k
                    break
            if img_key is None:
                img_key = keys[0]
            self.img_key = img_key
            self.length = f[img_key].shape[0]

            # Find mass and label keys if labeled
            if labeled:
                self.mass_key = None
                self.label_key = None
                for k in ["m0", "mass", "m", "y_mass", "target_mass"]:
                    if k in keys:
                        self.mass_key = k
                        break
                for k in ["label", "labels", "y", "y_label", "target_label", "pid"]:
                    if k in keys:
                        self.label_key = k
                        break

    def _get_file(self) -> h5py.File:
        """Return an open file handle, opening lazily if needed."""
        if self._file is None or not self._file.id.valid:
            self._file = h5py.File(self.filepath, "r")
        return self._file

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, idx: int):
        f = self._get_file()
        img = f[self.img_key][idx]  # shape: (C, H, W) or (H, W)
        img = torch.from_numpy(np.array(img, dtype=np.float32))
        if img.ndim == 2:
            img = img.unsqueeze(0)  # (1, H, W)

        if self.transform is not None:
            img = self.transform(img)

        if self.labeled:
            mass = float(f[self.mass_key][idx]) if self.mass_key else 0.0
            label = int(f[self.label_key][idx]) if self.label_key else 0
            return img, torch.tensor(mass, dtype=torch.float32), torch.tensor(label, dtype=torch.long)
        else:
            return (img,)

    def __del__(self):
        if self._file is not None and self._file.id.valid:
            self._file.close()


class SyntheticDataset(Dataset):
    """
    Synthetic dataset for demonstration when HDF5 files are not available.

    Generates random particle-like images with random masses and classes.

    Parameters
    ----------
    n_samples : int
        Number of synthetic samples to generate.
    img_size : int
        Spatial resolution of each image.
    in_chans : int
        Number of image channels.
    num_classes : int
        Number of class labels.
    labeled : bool
        If True, returns (image, mass, label). Else returns (image,).
    """

    def __init__(
        self,
        n_samples: int = 1000,
        img_size: int = 64,
        in_chans: int = 8,
        num_classes: int = 2,
        labeled: bool = True,
        transform=None,
    ):
        self.n_samples = n_samples
        self.img_size = img_size
        self.in_chans = in_chans
        self.num_classes = num_classes
        self.labeled = labeled
        self.transform = transform

        rng = np.random.default_rng(SEED)
        # Sparse energy deposits (most pixels near zero, few hot spots)
        self.images = rng.exponential(0.1, (n_samples, in_chans, img_size, img_size)).astype(np.float32)
        self.masses = rng.uniform(0.1, 200.0, n_samples).astype(np.float32)
        self.labels = rng.integers(0, num_classes, n_samples).astype(np.int64)

    def __len__(self) -> int:
        return self.n_samples

    def __getitem__(self, idx: int):
        img = torch.from_numpy(self.images[idx])
        if self.transform is not None:
            img = self.transform(img)
        if self.labeled:
            return img, torch.tensor(self.masses[idx]), torch.tensor(self.labels[idx])
        return (img,)


# ----------------------------------------------------------
# Load or create datasets
# ----------------------------------------------------------

print("\n--- Loading Datasets ---")
if LABELED_FILE.exists():
    print(f"Found labeled file: {LABELED_FILE}")
    inspect_hdf5(str(LABELED_FILE))
    raw_dataset = LazyHDF5Dataset(LABELED_FILE, labeled=True)
    print(f"Labeled dataset size: {len(raw_dataset)}")
    USING_SYNTHETIC = False
else:
    print(f"HDF5 file not found at {LABELED_FILE}")
    print("Using synthetic fallback dataset (1000 samples, 64×64, 8-channel)")
    raw_dataset = SyntheticDataset(
        n_samples=1000, img_size=64, in_chans=IN_CHANS, num_classes=NUM_CLASSES, labeled=True
    )
    USING_SYNTHETIC = True
    print(f"Synthetic dataset size: {len(raw_dataset)}")

if UNLABELED_FILE.exists():
    print(f"\nFound unlabeled file: {UNLABELED_FILE}")
    inspect_hdf5(str(UNLABELED_FILE))
    unlabeled_dataset_raw = LazyHDF5Dataset(UNLABELED_FILE, labeled=False)
    print(f"Unlabeled dataset size: {len(unlabeled_dataset_raw)}")
else:
    print(f"\nUnlabeled HDF5 not found. Creating small synthetic unlabeled set.")
    unlabeled_dataset_raw = SyntheticDataset(n_samples=200, img_size=64, in_chans=IN_CHANS, labeled=False)

## Section 3: Data Preprocessing & Augmentation

In [ ]:
# ============================================================
# Section 3: Data Preprocessing & Augmentation
# Physics-inspired transforms + optional augmentations
# ============================================================

import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt


class PhysicsPreprocess(nn.Module):
    """
    Physics-inspired preprocessing pipeline for detector images.

    Steps:
    1. Log-compress: x = log(1 + x)  (handles wide energy dynamic range)
    2. Per-image min-max normalization to [0, 1]
    3. Resize to (img_size, img_size)
    4. Ensure IN_CHANS channels (tile if fewer channels)
    """

    def __init__(self, img_size: int = 64, in_chans: int = 8):
        super().__init__()
        self.img_size = img_size
        self.in_chans = in_chans

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (C, H, W)
        # 1. Log-compress
        x = torch.log1p(x.clamp(min=0.0))

        # 2. Per-image min-max normalization
        x_min = x.min()
        x_max = x.max()
        if x_max > x_min:
            x = (x - x_min) / (x_max - x_min + 1e-8)
        else:
            x = torch.zeros_like(x)

        # 3. Resize spatial dimensions
        if x.shape[-1] != self.img_size or x.shape[-2] != self.img_size:
            x = TF.resize(x, [self.img_size, self.img_size], antialias=True)

        # 4. Ensure correct number of channels
        c = x.shape[0]
        if c < self.in_chans:
            repeats = (self.in_chans + c - 1) // c
            x = x.repeat(repeats, 1, 1)[: self.in_chans]
        elif c > self.in_chans:
            x = x[: self.in_chans]

        return x


class AugmentTransform(nn.Module):
    """
    Optional data augmentation for detector images.

    Augmentations:
    - Random 90° rotation increments
    - Random horizontal/vertical flips
    - Gaussian noise injection (σ=0.01)

    All are applied with 50% probability.
    """

    def __init__(self, p: float = 0.5, noise_std: float = 0.01):
        super().__init__()
        self.p = p
        self.noise_std = noise_std

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if random.random() < self.p:
            k = random.randint(0, 3)
            x = torch.rot90(x, k, dims=[-2, -1])
        if random.random() < self.p:
            x = TF.hflip(x)
        if random.random() < self.p:
            x = TF.vflip(x)
        if random.random() < self.p:
            x = x + torch.randn_like(x) * self.noise_std
            x = x.clamp(0.0, 1.0)
        return x


class TrainTransform(nn.Module):
    """Combined preprocessing + augmentation for training."""
    def __init__(self, img_size=64, in_chans=8):
        super().__init__()
        self.preprocess = PhysicsPreprocess(img_size, in_chans)
        self.augment = AugmentTransform()

    def forward(self, x):
        return self.augment(self.preprocess(x))


class ValTransform(nn.Module):
    """Preprocessing only (no augmentation) for validation/test."""
    def __init__(self, img_size=64, in_chans=8):
        super().__init__()
        self.preprocess = PhysicsPreprocess(img_size, in_chans)

    def forward(self, x):
        return self.preprocess(x)


# ----------------------------------------------------------
# Wrap datasets with transforms and create DataLoaders
# ----------------------------------------------------------

class TransformedDataset(Dataset):
    """Wraps a base dataset and applies a transform on __getitem__."""
    def __init__(self, base_dataset: Dataset, transform=None):
        self.base = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        item = self.base[idx]
        if self.transform is not None:
            img = self.transform(item[0])
            return (img,) + item[1:]
        return item


# Train/Val split (fixed seed for reproducibility)
n_total = len(raw_dataset)
n_train = int(n_total * TRAIN_FRAC)
n_val = n_total - n_train

generator = torch.Generator().manual_seed(SEED)
train_raw, val_raw = random_split(raw_dataset, [n_train, n_val], generator=generator)

print(f"Train samples: {n_train} | Val samples: {n_val}")

# Apply transforms
train_dataset = TransformedDataset(train_raw, TrainTransform(IMG_SIZE, IN_CHANS))
val_dataset = TransformedDataset(val_raw, ValTransform(IMG_SIZE, IN_CHANS))

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


# ----------------------------------------------------------
# Visualize sample images
# ----------------------------------------------------------

def plot_sample_images_raw(dataset: Dataset, n: int = 8, title: str = "Sample Images") -> None:
    """Visualize a grid of sample detector images (first channel shown)."""
    fig, axes = plt.subplots(1, n, figsize=(2 * n, 2.5))
    for i, ax in enumerate(axes):
        if i >= len(dataset):
            break
        item = dataset[i]
        img = item[0]
        if img.ndim == 3:
            img = img[0]  # show first channel
        ax.imshow(img.numpy(), cmap="hot", aspect="auto")
        ax.axis("off")
        if len(item) >= 3:
            mass, label = item[1].item(), item[2].item()
            ax.set_title(f"m={mass:.1f}\ncls={int(label)}", fontsize=7)
    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Show unlabeled samples (first 8)
print("\n--- Unlabeled samples (log-compressed, ch 0) ---")
unlabeled_vis = TransformedDataset(unlabeled_dataset_raw, ValTransform(IMG_SIZE, IN_CHANS))
plot_sample_images_raw(unlabeled_vis, n=min(8, len(unlabeled_vis)), title="Unlabeled Samples (Channel 0)")

# Show labeled samples (first 8 from validation)
print("\n--- Labeled validation samples (log-compressed, ch 0) ---")
plot_sample_images_raw(val_dataset, n=min(8, len(val_dataset)), title="Labeled Samples with Mass & Class Annotations")

## Section 4: Model Architectures

Three architectures are compared:

| Model | Attention | Complexity | Description |
|-------|-----------|------------|-------------|
| **StandardViT** | Softmax self-attn | O(N²·d) | Classic ViT with learnable pos. embeddings |
| **LinearAttentionViT** | Cross-Covariance (XCA) | O(N·d²) | XCiT-style linear attention with LPI |
| **HybridCNNViT** | CNN + Softmax | Mixed | CNN stem tokenizes, transformer refines |

All three share: EMBED_DIM=256, NUM_HEADS=8, DEPTH=10, stochastic depth decay [0→0.1].

Architecture reference diagrams: `../images/`

In [ ]:
# ============================================================
# Section 4: Model Architectures
# StandardViT, LinearAttentionViT, HybridCNNViT
# ============================================================

import math
from functools import partial

import torch.nn.functional as F


# ----------------------------------------------------------
# Shared building blocks
# ----------------------------------------------------------

class DropPath(nn.Module):
    """
    Stochastic Depth (Drop Path) regularization for deep networks.

    Randomly drops entire residual branches during training,
    equivalent to training an ensemble of shallower networks.

    Reference: "Deep Networks with Stochastic Depth" (Huang et al., 2016)
    """

    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor = torch.floor(random_tensor + keep_prob)
        return x * random_tensor / keep_prob


class PatchEmbed(nn.Module):
    """
    Image-to-patch embedding using a strided convolution.

    Splits the image into non-overlapping patches and projects each
    patch to EMBED_DIM via a single Conv2d (equivalent to a linear
    projection of flattened patches).

    Parameters
    ----------
    img_size : int
        Input image resolution (assumed square).
    patch_size : int
        Patch edge length in pixels.
    in_chans : int
        Number of input image channels.
    embed_dim : int
        Output embedding dimension.
    """

    def __init__(self, img_size=64, patch_size=8, in_chans=8, embed_dim=256):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, H, W) -> (B, N, D)
        x = self.proj(x)                          # (B, D, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)          # (B, N, D)
        return x


class MLP(nn.Module):
    """
    Two-layer feed-forward network with GELU activation and dropout.

    Used inside transformer blocks (FFN / MLP sublayer).
    """

    def __init__(self, in_features: int, hidden_features: int = None, dropout: float = 0.0):
        super().__init__()
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, in_features)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


def _make_head(in_dim: int, out_dim: int, dropout: float = 0.1) -> nn.Sequential:
    """Build a two-layer MLP head: Linear → GELU → Dropout → Linear."""
    return nn.Sequential(
        nn.Linear(in_dim, in_dim // 2),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(in_dim // 2, out_dim),
    )


# ----------------------------------------------------------
# 4.1  Standard Vision Transformer (ViT)
# ----------------------------------------------------------

class MultiHeadSelfAttention(nn.Module):
    """
    Standard multi-head self-attention with softmax: O(N²·d).

    Complexity: O(N² · d) per layer, where N = number of tokens,
    d = head dimension. Scales quadratically with sequence length.
    """

    def __init__(self, dim: int, num_heads: int = 8, dropout: float = 0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)           # (3, B, H, N, d)
        q, k, v = qkv.unbind(0)                      # each (B, H, N, d)
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, H, N, N)
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, D)
        return self.proj_drop(self.proj(x))


class ViTBlock(nn.Module):
    """
    Standard pre-norm ViT transformer block.

    Architecture: x = x + Attn(LN(x)); x = x + FFN(LN(x))
    Includes stochastic depth (DropPath) regularization.
    """

    def __init__(self, dim: int, num_heads: int, mlp_ratio: float, dropout: float, drop_path: float):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadSelfAttention(dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, int(dim * mlp_ratio), dropout)
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class StandardViT(nn.Module):
    """
    Standard Vision Transformer (ViT) with quadratic self-attention.

    Reference: Dosovitskiy et al., "An Image is Worth 16×16 Words", ICLR 2021.

    Architecture:
    - PatchEmbed (Conv2d stride=patch_size)
    - Learnable positional embeddings
    - DEPTH transformer blocks with pre-norm and stochastic depth
    - Global average pooling
    - Regression head (mass) and classification head (class)
    """

    def __init__(
        self,
        img_size=IMG_SIZE,
        patch_size=PATCH_SIZE,
        in_chans=IN_CHANS,
        embed_dim=EMBED_DIM,
        depth=DEPTH,
        num_heads=NUM_HEADS,
        mlp_ratio=MLP_RATIO,
        dropout=DROPOUT,
        num_classes=NUM_CLASSES,
        drop_path_rate=0.1,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        n_patches = self.patch_embed.n_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        # Stochastic depth decay rule (linear from 0 to drop_path_rate)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList([
            ViTBlock(embed_dim, num_heads, mlp_ratio, dropout, dpr[i])
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        # Output heads
        self.regression_head = _make_head(embed_dim, 1, dropout)
        self.classification_head = _make_head(embed_dim, num_classes, dropout)

        # Weight initialization
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def count_params(self) -> int:
        """Return total number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def freeze_encoder(self) -> None:
        """Freeze all parameters except output heads."""
        for name, param in self.named_parameters():
            if "regression_head" not in name and "classification_head" not in name:
                param.requires_grad = False

    def unfreeze_encoder(self) -> None:
        """Unfreeze all parameters."""
        for param in self.parameters():
            param.requires_grad = True

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        """Encode image to (B, embed_dim) feature vector."""
        x = self.patch_embed(x)
        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x.mean(dim=1)  # global average pooling

    def forward(self, x: torch.Tensor):
        """
        Parameters
        ----------
        x : Tensor (B, C, H, W)

        Returns
        -------
        mass_pred : Tensor (B, 1)
        class_logits : Tensor (B, num_classes)
        """
        feat = self.forward_features(x)
        return self.regression_head(feat), self.classification_head(feat)


# ----------------------------------------------------------
# 4.2  Linear Attention Vision Transformer (XCiT-style)
# ----------------------------------------------------------

class CrossCovarianceAttention(nn.Module):
    """
    Cross-Covariance Attention (XCA) from XCiT.

    Complexity: O(N·d²) — operates in feature (channel) space.

    Instead of the N×N token-attention matrix, XCA computes a d×d
    cross-covariance matrix between L2-normalised Q and K:
        A = (Q^T K) / τ   (d×d, temperature-scaled)
        out = softmax(A) V^T   transposed back to (B, N, d)

    This scales linearly with N (number of tokens) rather than
    quadratically, making it efficient for high-resolution inputs.

    Reference: El-Nouby et al., "XCiT: Cross-Covariance Image
    Transformers", NeurIPS 2021.
    """

    def __init__(self, dim: int, num_heads: int = 8, dropout: float = 0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))
        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)          # (3, B, H, N, d)
        q, k, v = qkv.unbind(0)                     # each (B, H, N, d)

        # L2-normalise along the token dimension (dim=-2: N)
        q = F.normalize(q, dim=-2)
        k = F.normalize(k, dim=-2)

        # Cross-covariance attention: (B, H, d, d)
        attn = (q.transpose(-2, -1) @ k) * self.temperature  # (B, H, d, d)
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        # Output: (B, H, N, d)
        x = (v @ attn).transpose(1, 2).reshape(B, N, D)
        return self.proj_drop(self.proj(x))


class LocalPatchInteraction(nn.Module):
    """
    Local Patch Interaction (LPI) module from XCiT.

    Two depth-wise 3×3 convolutions with BatchNorm and GELU activation,
    applied in the spatial domain to capture local patch correlations
    that global attention may miss.

    The token sequence (B, N, D) is reshaped to a 2-D feature map
    (B, D, H/P, W/P) for convolution, then reshaped back.
    """

    def __init__(self, dim: int, n_patches_side: int):
        super().__init__()
        self.n = n_patches_side  # patches per side
        self.conv1 = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)
        self.bn = nn.BatchNorm2d(dim)
        self.act = nn.GELU()
        self.conv2 = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, D = x.shape
        # (B, N, D) -> (B, D, n, n)
        x2d = x.transpose(1, 2).reshape(B, D, self.n, self.n)
        x2d = self.act(self.bn(self.conv1(x2d)))
        x2d = self.conv2(x2d)
        return x2d.reshape(B, D, N).transpose(1, 2)  # (B, N, D)


class XCiTBlock(nn.Module):
    """
    XCiT transformer block combining XCA, LPI, and FFN.

    Architecture: 
        x = x + XCA(LN(x))       # global linear attention
        x = x + LPI(LN(x))       # local patch interaction
        x = x + FFN(LN(x))       # feed-forward network
    """

    def __init__(self, dim: int, num_heads: int, mlp_ratio: float, dropout: float,
                 drop_path: float, n_patches_side: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = CrossCovarianceAttention(dim, num_heads, dropout)
        self.norm_lpi = nn.LayerNorm(dim)
        self.lpi = LocalPatchInteraction(dim, n_patches_side)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, int(dim * mlp_ratio), dropout)
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.lpi(self.norm_lpi(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class LinearAttentionViT(nn.Module):
    """
    Linear Attention Vision Transformer using Cross-Covariance Attention (XCA).

    Achieves O(N·d²) complexity per layer by computing attention in the
    channel space (d×d) rather than the token space (N×N), making it
    especially efficient when N >> d (many small patches).

    Reference: El-Nouby et al., "XCiT: Cross-Covariance Image
    Transformers", NeurIPS 2021.
    """

    def __init__(
        self,
        img_size=IMG_SIZE,
        patch_size=PATCH_SIZE,
        in_chans=IN_CHANS,
        embed_dim=EMBED_DIM,
        depth=DEPTH,
        num_heads=NUM_HEADS,
        mlp_ratio=MLP_RATIO,
        dropout=DROPOUT,
        num_classes=NUM_CLASSES,
        drop_path_rate=0.1,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        n_patches = self.patch_embed.n_patches
        n_side = img_size // patch_size
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList([
            XCiTBlock(embed_dim, num_heads, mlp_ratio, dropout, dpr[i], n_side)
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        self.regression_head = _make_head(embed_dim, 1, dropout)
        self.classification_head = _make_head(embed_dim, num_classes, dropout)

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def count_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def freeze_encoder(self) -> None:
        for name, param in self.named_parameters():
            if "regression_head" not in name and "classification_head" not in name:
                param.requires_grad = False

    def unfreeze_encoder(self) -> None:
        for param in self.parameters():
            param.requires_grad = True

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed(x)
        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x.mean(dim=1)

    def forward(self, x: torch.Tensor):
        feat = self.forward_features(x)
        return self.regression_head(feat), self.classification_head(feat)


# ----------------------------------------------------------
# 4.3  Hybrid CNN + Vision Transformer
# ----------------------------------------------------------

class ConvBlock(nn.Module):
    """
    Single CNN block: Conv2d → BatchNorm → GELU → MaxPool2d(2).
    Used to build the CNN backbone in HybridCNNViT.
    """

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = nn.GELU()
        self.pool = nn.MaxPool2d(2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.pool(self.act(self.bn(self.conv(x))))


class HybridCNNViT(nn.Module):
    """
    Hybrid CNN + Vision Transformer architecture.

    The CNN backbone extracts hierarchical local features and produces
    a spatial feature map, which is then flattened into a token sequence
    for the transformer encoder to model long-range dependencies.

    CNN backbone (3 blocks, each halving spatial resolution):
        Block 1: IN_CHANS →  64 channels  (64→32)
        Block 2:  64      → 128 channels  (32→16)
        Block 3: 128      → EMBED_DIM     (16→ 8)
    Token sequence: 8×8 = 64 tokens of dimension EMBED_DIM
    Transformer: DEPTH//2 standard blocks (same as ViT)
    """

    def __init__(
        self,
        img_size=IMG_SIZE,
        in_chans=IN_CHANS,
        embed_dim=EMBED_DIM,
        depth=DEPTH,
        num_heads=NUM_HEADS,
        mlp_ratio=MLP_RATIO,
        dropout=DROPOUT,
        num_classes=NUM_CLASSES,
        drop_path_rate=0.1,
    ):
        super().__init__()
        # CNN backbone: 3 downsampling blocks
        self.cnn = nn.Sequential(
            ConvBlock(in_chans, 64),
            ConvBlock(64, 128),
            ConvBlock(128, embed_dim),
        )
        # After 3 max-pool-2: spatial dim = img_size // 8
        spatial = img_size // 8
        n_tokens = spatial * spatial
        self.n_tokens = n_tokens

        self.pos_embed = nn.Parameter(torch.zeros(1, n_tokens, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        # Transformer encoder (half the depth of pure ViT)
        transformer_depth = max(depth // 2, 2)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, transformer_depth)]
        self.blocks = nn.ModuleList([
            ViTBlock(embed_dim, num_heads, mlp_ratio, dropout, dpr[i])
            for i in range(transformer_depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        self.regression_head = _make_head(embed_dim, 1, dropout)
        self.classification_head = _make_head(embed_dim, num_classes, dropout)

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, (nn.LayerNorm, nn.BatchNorm2d)):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def count_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def freeze_encoder(self) -> None:
        for name, param in self.named_parameters():
            if "regression_head" not in name and "classification_head" not in name:
                param.requires_grad = False

    def unfreeze_encoder(self) -> None:
        for param in self.parameters():
            param.requires_grad = True

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        # CNN: (B, C, H, W) → (B, D, H', W')
        x = self.cnn(x)
        B, D, H, W = x.shape
        # Flatten spatial: (B, D, H', W') → (B, H'*W', D)
        x = x.flatten(2).transpose(1, 2)
        x = self.pos_drop(x + self.pos_embed[:, :x.size(1)])
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x.mean(dim=1)

    def forward(self, x: torch.Tensor):
        feat = self.forward_features(x)
        return self.regression_head(feat), self.classification_head(feat)


# ----------------------------------------------------------
# Quick shape tests and parameter counts
# ----------------------------------------------------------

print("=" * 60)
print("Model Architecture Summary")
print("=" * 60)

_dummy = torch.zeros(2, IN_CHANS, IMG_SIZE, IMG_SIZE)
for ModelClass in [StandardViT, LinearAttentionViT, HybridCNNViT]:
    model = ModelClass()
    mass, cls = model(_dummy)
    params = model.count_params()
    print(f"\n{ModelClass.__name__}:")
    print(f"  mass_pred shape : {mass.shape}")
    print(f"  class_logits shape: {cls.shape}")
    print(f"  Trainable params: {params:,} ({params/1e6:.2f}M)")
    del model
print("=" * 60)

## Section 5: Training Utilities

In [ ]:
# ============================================================
# Section 5: Training Utilities
# Mixed-precision training, evaluation, early stopping,
# cosine LR scheduling, and model checkpointing
# ============================================================

import copy
from tqdm.auto import tqdm
from torch.cuda.amp import GradScaler, autocast


# ----------------------------------------------------------
# Loss function
# ----------------------------------------------------------

def compute_loss(
    mass_pred: torch.Tensor,
    mass_true: torch.Tensor,
    class_logits: torch.Tensor,
    class_true: torch.Tensor,
    regression_weight: float = REGRESSION_LAMBDA,
) -> tuple:
    """
    Combined classification + regression loss.

    Total Loss = CrossEntropy(class) + regression_weight * MSE(mass)

    Parameters
    ----------
    mass_pred : Tensor (B, 1)
    mass_true : Tensor (B,)
    class_logits : Tensor (B, num_classes)
    class_true : Tensor (B,) of dtype long
    regression_weight : float

    Returns
    -------
    total_loss, mse_loss, ce_loss (all scalar Tensors)
    """
    mse = F.mse_loss(mass_pred.squeeze(1), mass_true)
    ce = F.cross_entropy(class_logits, class_true)
    total = ce + regression_weight * mse
    return total, mse, ce


# ----------------------------------------------------------
# Training epoch
# ----------------------------------------------------------

def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    regression_weight: float = REGRESSION_LAMBDA,
) -> dict:
    """
    Run one training epoch with mixed precision (AMP) and gradient clipping.

    Parameters
    ----------
    model : nn.Module
    loader : DataLoader  (yields img, mass, label)
    optimizer : torch.optim.Optimizer
    scaler : GradScaler  (for AMP; ignored gracefully on CPU)
    regression_weight : float  (λ for MSE loss)

    Returns
    -------
    dict with keys: loss, mse, ce  (epoch averages)
    """
    model.train()
    total_loss = total_mse = total_ce = 0.0
    n_batches = len(loader)

    for imgs, masses, labels in tqdm(loader, desc="  train", leave=False):
        imgs = imgs.to(DEVICE)
        masses = masses.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        # Mixed precision forward pass
        with autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            mass_pred, class_logits = model(imgs)
            loss, mse, ce = compute_loss(mass_pred, masses, class_logits, labels, regression_weight)

        # Backward + gradient clipping + optimizer step
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        total_mse += mse.item()
        total_ce += ce.item()

    return {"loss": total_loss / n_batches, "mse": total_mse / n_batches, "ce": total_ce / n_batches}


# ----------------------------------------------------------
# Evaluation
# ----------------------------------------------------------

@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader) -> dict:
    """
    Evaluate model on the given DataLoader.

    Parameters
    ----------
    model : nn.Module
    loader : DataLoader  (yields img, mass, label)

    Returns
    -------
    dict with keys: mass_pred, mass_true, class_pred, class_true,
                    loss, mse, ce
    """
    model.eval()
    all_mass_pred, all_mass_true = [], []
    all_class_pred, all_class_true = [], []
    total_loss = total_mse = total_ce = 0.0
    n_batches = len(loader)

    for imgs, masses, labels in tqdm(loader, desc="  eval ", leave=False):
        imgs = imgs.to(DEVICE)
        masses = masses.to(DEVICE)
        labels = labels.to(DEVICE)

        mass_pred, class_logits = model(imgs)
        loss, mse, ce = compute_loss(mass_pred, masses, class_logits, labels)

        total_loss += loss.item()
        total_mse += mse.item()
        total_ce += ce.item()

        all_mass_pred.append(mass_pred.squeeze(1).cpu())
        all_mass_true.append(masses.cpu())
        all_class_pred.append(class_logits.argmax(dim=1).cpu())
        all_class_true.append(labels.cpu())

    return {
        "mass_pred": torch.cat(all_mass_pred).numpy(),
        "mass_true": torch.cat(all_mass_true).numpy(),
        "class_pred": torch.cat(all_class_pred).numpy(),
        "class_true": torch.cat(all_class_true).numpy(),
        "loss": total_loss / n_batches,
        "mse": total_mse / n_batches,
        "ce": total_ce / n_batches,
    }


# ----------------------------------------------------------
# Early stopping
# ----------------------------------------------------------

class EarlyStopping:
    """
    Early stopping based on validation metric (lower is better).

    Stops training when the monitored metric does not improve
    for `patience` consecutive epochs.

    Parameters
    ----------
    patience : int
        Number of epochs with no improvement before stopping.
    min_delta : float
        Minimum change to qualify as improvement.
    """

    def __init__(self, patience: int = 5, min_delta: float = 1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score: float = float("inf")
        self.counter: int = 0
        self.should_stop: bool = False

    def step(self, score: float) -> bool:
        """
        Call each epoch with the validation metric.

        Returns True if training should stop.
        """
        if score < self.best_score - self.min_delta:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop

    def reset(self) -> None:
        """Reset state for a new training run."""
        self.best_score = float("inf")
        self.counter = 0
        self.should_stop = False


print("Training utilities defined:")
print("  - compute_loss (CE + λ·MSE)")
print("  - train_epoch (AMP + gradient clipping)")
print("  - evaluate_model (no_grad evaluation)")
print("  - EarlyStopping (patience-based)")

## Section 6: Evaluation Metrics

In [ ]:
# ============================================================
# Section 6: Evaluation Metrics
# Full suite of classification + regression metrics
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)


def compute_metrics(eval_results: dict) -> dict:
    """
    Compute comprehensive classification and regression metrics.

    Parameters
    ----------
    eval_results : dict
        Output of evaluate_model(). Must have keys:
        mass_pred, mass_true, class_pred, class_true.

    Returns
    -------
    dict with keys:
        accuracy, f1, precision, recall, confusion_matrix  (classification)
        mse, mae, r2                                        (regression)
    """
    y_true = eval_results["class_true"]
    y_pred = eval_results["class_pred"]
    m_true = eval_results["mass_true"]
    m_pred = eval_results["mass_pred"]

    metrics = {
        # Classification
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        # Regression
        "mse": mean_squared_error(m_true, m_pred),
        "mae": mean_absolute_error(m_true, m_pred),
        "r2": r2_score(m_true, m_pred),
    }
    return metrics


def print_metrics(metrics: dict, title: str = "Metrics") -> None:
    """Pretty-print a metrics dictionary."""
    print(f"\n{'='*50}")
    print(f"  {title}")
    print(f"{'='*50}")
    print(f"  Classification:")
    print(f"    Accuracy  : {metrics['accuracy']:.4f}")
    print(f"    F1 (macro): {metrics['f1']:.4f}")
    print(f"    Precision : {metrics['precision']:.4f}")
    print(f"    Recall    : {metrics['recall']:.4f}")
    print(f"  Regression:")
    print(f"    MSE       : {metrics['mse']:.4f}")
    print(f"    MAE       : {metrics['mae']:.4f}")
    print(f"    R²        : {metrics['r2']:.4f}")
    print(f"{'='*50}")


print("Metrics functions defined: compute_metrics(), print_metrics()")

## Section 7: Visualization Tools

In [ ]:
# ============================================================
# Section 7: Visualization Tools
# Plotting functions for training curves, metrics, and images
# ============================================================

import seaborn as sns
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 100,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "font.size": 10,
})


def plot_training_curves(history: dict, title: str = "Training Curves") -> None:
    """
    Plot loss and accuracy vs epoch for a single model.

    Parameters
    ----------
    history : dict
        Keys: train_loss, val_loss, val_acc (lists, one value per epoch).
    title : str
        Figure title.
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history["train_loss"]) + 1)

    axes[0].plot(epochs, history["train_loss"], label="Train Loss", linewidth=2)
    axes[0].plot(epochs, history["val_loss"], label="Val Loss", linewidth=2, linestyle="--")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title(f"{title} — Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history["val_acc"], label="Val Accuracy", linewidth=2, color="green")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title(f"{title} — Validation Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_mass_scatter(
    true_mass: np.ndarray,
    pred_mass: np.ndarray,
    title: str = "Mass Prediction",
) -> None:
    """
    Scatter plot of true vs predicted particle mass with R² annotation.

    Parameters
    ----------
    true_mass : array (N,)
    pred_mass : array (N,)
    title : str
    """
    r2 = r2_score(true_mass, pred_mass)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(true_mass, pred_mass, alpha=0.3, s=10, edgecolors="none")
    lims = [min(true_mass.min(), pred_mass.min()), max(true_mass.max(), pred_mass.max())]
    ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")
    ax.set_xlabel("True Mass")
    ax.set_ylabel("Predicted Mass")
    ax.set_title(f"{title}\n$R^2 = {r2:.4f}$")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(
    cm: np.ndarray,
    class_names: list = None,
    title: str = "Confusion Matrix",
) -> None:
    """
    Heatmap of the confusion matrix using seaborn.

    Parameters
    ----------
    cm : array (n_classes, n_classes)
    class_names : list of str
    title : str
    """
    if class_names is None:
        class_names = [str(i) for i in range(len(cm))]
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def plot_sample_images(
    dataset: Dataset,
    n: int = 8,
    title: str = "Sample Images",
) -> None:
    """
    Visualize a grid of detector images (first channel shown in false color).

    Parameters
    ----------
    dataset : Dataset  (returns (img, ...) tuples)
    n : int  number of images to show
    title : str
    """
    fig, axes = plt.subplots(1, n, figsize=(2 * n, 2.5))
    for i, ax in enumerate(axes):
        if i >= len(dataset):
            break
        item = dataset[i]
        img = item[0]
        if img.ndim == 3:
            img = img[0]
        ax.imshow(img.numpy(), cmap="hot", aspect="auto")
        ax.axis("off")
        if len(item) >= 3:
            ax.set_title(f"m={item[1].item():.1f}\ncls={int(item[2])}", fontsize=7)
    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


def plot_comparison_curves(
    all_histories: dict,
    metric_name: str = "val_loss",
) -> None:
    """
    Overlay training curves from multiple models for comparison.

    Parameters
    ----------
    all_histories : dict  {model_name: history_dict}
    metric_name : str  key in history_dict to plot
    """
    fig, ax = plt.subplots(figsize=(9, 5))
    for model_name, history in all_histories.items():
        vals = history.get(metric_name, [])
        epochs = range(1, len(vals) + 1)
        ax.plot(epochs, vals, linewidth=2, label=model_name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric_name.replace("_", " ").title())
    ax.set_title(f"Comparison: {metric_name.replace('_', ' ').title()}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


print("Visualization functions defined:")
for fn in ["plot_training_curves", "plot_mass_scatter", "plot_confusion_matrix",
           "plot_sample_images", "plot_comparison_curves"]:
    print(f"  - {fn}()")

## Section 8: Experiments

Each of the three models is trained with **identical** hyperparameters:
- Optimizer: AdamW (lr=3e-4, weight_decay=1e-4)
- Scheduler: CosineAnnealingLR over EPOCHS
- Loss: CE + λ·MSE (λ=1.0)
- Same 80/20 train/val split
- Mixed precision training (CUDA) or standard (CPU/MPS)
- Early stopping (patience=5, monitors val MSE)

In [ ]:
# ============================================================
# Section 8: Experiments
# Full training pipeline for each architecture
# ============================================================

import gc
import tempfile


def run_experiment(
    model_class,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = LR,
    weight_decay: float = WEIGHT_DECAY,
    regression_weight: float = REGRESSION_LAMBDA,
    patience: int = 5,
) -> dict:
    """
    Full training pipeline for a single model architecture.

    Steps:
    1. Instantiate model and move to DEVICE
    2. Setup AdamW optimizer + CosineAnnealingLR scheduler
    3. Train for `epochs` with mixed precision, early stopping,
       and best-model checkpointing (based on val MSE)
    4. Load best checkpoint and run final evaluation
    5. Return results dict with all metrics and histories

    Parameters
    ----------
    model_class : type  (StandardViT, LinearAttentionViT, or HybridCNNViT)
    model_name : str    display name for logging
    train_loader, val_loader : DataLoader
    epochs : int
    lr : float          learning rate
    weight_decay : float
    regression_weight : float  λ for MSE loss
    patience : int      early stopping patience

    Returns
    -------
    dict with keys: model_name, history, final_metrics, train_time, params,
                    peak_gpu_mem_gb
    """
    print(f"\n{'='*60}")
    print(f"  Experiment: {model_name}")
    print(f"{'='*60}")

    seed_everything(SEED)  # Same initialization for fairness
    model = model_class().to(DEVICE)
    params = model.count_params()
    print(f"  Parameters: {params:,} ({params/1e6:.2f}M)")
    print(f"  Device: {DEVICE}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = GradScaler(enabled=(DEVICE.type == "cuda"))
    early_stopper = EarlyStopping(patience=patience)

    # Checkpointing: save best model state to a temp file
    best_state = None
    best_val_mse = float("inf")

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_mse": [],
        "val_acc": [],
        "lr": [],
    }

    start_time = time.time()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)

    for epoch in range(1, epochs + 1):
        current_lr = optimizer.param_groups[0]["lr"]

        # --- Train ---
        train_stats = train_epoch(model, train_loader, optimizer, scaler, regression_weight)

        # --- Evaluate ---
        val_results = evaluate_model(model, val_loader)
        val_metrics = compute_metrics(val_results)

        # --- Scheduler step ---
        scheduler.step()

        # --- Logging ---
        history["train_loss"].append(train_stats["loss"])
        history["val_loss"].append(val_results["loss"])
        history["val_mse"].append(val_metrics["mse"])
        history["val_acc"].append(val_metrics["accuracy"])
        history["lr"].append(current_lr)

        print(
            f"  Epoch {epoch:2d}/{epochs} | "
            f"Train Loss: {train_stats['loss']:.4f} | "
            f"Val Loss: {val_results['loss']:.4f} | "
            f"Val Acc: {val_metrics['accuracy']:.4f} | "
            f"Val MSE: {val_metrics['mse']:.4f} | "
            f"LR: {current_lr:.2e}"
        )

        # --- Checkpointing (best val MSE) ---
        if val_metrics["mse"] < best_val_mse:
            best_val_mse = val_metrics["mse"]
            best_state = copy.deepcopy(model.state_dict())

        # --- Early stopping ---
        if early_stopper.step(val_metrics["mse"]):
            print(f"  Early stopping at epoch {epoch} (patience={patience})")
            break

    train_time = time.time() - start_time

    # Peak GPU memory
    if DEVICE.type == "cuda":
        peak_mem_gb = torch.cuda.max_memory_allocated(DEVICE) / 1e9
    else:
        peak_mem_gb = 0.0

    # Load best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"  Loaded best checkpoint (val MSE: {best_val_mse:.4f})")

    # Final evaluation
    print("  Running final evaluation...")
    final_results = evaluate_model(model, val_loader)
    final_metrics = compute_metrics(final_results)
    print_metrics(final_metrics, title=f"{model_name} — Final Validation Metrics")

    print(f"  Training time: {train_time:.1f}s | Peak GPU mem: {peak_mem_gb:.2f} GB")

    # Clean up
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "model_name": model_name,
        "history": history,
        "final_metrics": final_metrics,
        "final_results": final_results,
        "train_time": train_time,
        "params": params,
        "peak_gpu_mem_gb": peak_mem_gb,
    }


# ----------------------------------------------------------
# Run all experiments
# ----------------------------------------------------------

models_to_run = {
    "Standard ViT": StandardViT,
    "Linear Attention ViT": LinearAttentionViT,
    "Hybrid CNN+ViT": HybridCNNViT,
}

all_results = {}
for name, model_class in models_to_run.items():
    all_results[name] = run_experiment(
        model_class=model_class,
        model_name=name,
        train_loader=train_loader,
        val_loader=val_loader,
    )

print("\n\nAll experiments complete!")

## Section 9: Benchmark Comparison

Aggregated results across all three architectures.

In [ ]:
# ============================================================
# Section 9: Benchmark Comparison
# Summary table + comparison plots
# ============================================================

# ----------------------------------------------------------
# Comparison DataFrame
# ----------------------------------------------------------

rows = []
for name, res in all_results.items():
    m = res["final_metrics"]
    rows.append({
        "Model": name,
        "Accuracy": f"{m['accuracy']:.4f}",
        "F1": f"{m['f1']:.4f}",
        "MSE": f"{m['mse']:.4f}",
        "MAE": f"{m['mae']:.4f}",
        "R²": f"{m['r2']:.4f}",
        "Train Time (s)": f"{res['train_time']:.1f}",
        "Parameters": f"{res['params']:,}",
    })

df = pd.DataFrame(rows).set_index("Model")
print("\n" + "="*90)
print("Benchmark Comparison Table")
print("="*90)
print(df.to_string())
print("="*90)

# ----------------------------------------------------------
# Comparison plots
# ----------------------------------------------------------

# 1. Training loss curves (all 3 overlaid)
print("\n--- Comparison: Training Loss ---")
plot_comparison_curves(
    {name: res["history"] for name, res in all_results.items()},
    metric_name="train_loss",
)

# 2. Validation accuracy curves (all 3 overlaid)
print("--- Comparison: Validation Accuracy ---")
plot_comparison_curves(
    {name: res["history"] for name, res in all_results.items()},
    metric_name="val_acc",
)

# 3. Mass scatter plots (one per model in a row)
print("--- Mass Prediction Scatter Plots ---")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, res) in zip(axes, all_results.items()):
    fr = res["final_results"]
    r2 = res["final_metrics"]["r2"]
    ax.scatter(fr["mass_true"], fr["mass_pred"], alpha=0.3, s=8, edgecolors="none")
    lims = [
        min(fr["mass_true"].min(), fr["mass_pred"].min()),
        max(fr["mass_true"].max(), fr["mass_pred"].max()),
    ]
    ax.plot(lims, lims, "r--", linewidth=1.5)
    ax.set_xlabel("True Mass")
    ax.set_ylabel("Pred. Mass")
    ax.set_title(f"{name}\n$R^2={r2:.4f}$")
    ax.grid(True, alpha=0.3)
plt.suptitle("Mass Prediction: True vs Predicted", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# 4. Confusion matrices (one per model in a row)
print("--- Confusion Matrices ---")
class_names = [str(i) for i in range(NUM_CLASSES)]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, res) in zip(axes, all_results.items()):
    cm = res["final_metrics"]["confusion_matrix"]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{name}")
plt.suptitle("Confusion Matrices", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# 5. Bar chart comparing key metrics
print("--- Metric Bar Charts ---")
metric_keys = ["accuracy", "f1", "r2"]
fig, axes = plt.subplots(1, len(metric_keys), figsize=(14, 4))
for ax, metric in zip(axes, metric_keys):
    values = [all_results[name]["final_metrics"][metric] for name in all_results]
    bars = ax.bar(list(all_results.keys()), values,
                  color=["steelblue", "darkorange", "forestgreen"])
    ax.set_ylim(0, max(1.0, max(values) * 1.1))
    ax.set_ylabel(metric.upper())
    ax.set_title(f"Comparison: {metric.upper()}")
    ax.set_xticklabels(list(all_results.keys()), rotation=15, ha="right")
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
    ax.grid(True, axis="y", alpha=0.3)
plt.suptitle("Key Metric Comparison Across Architectures", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## Section 10: Final Results & Conclusions

In [ ]:
# ============================================================
# Section 10: Final Results & Conclusions
# ============================================================

# Print summary table once more
print("\n" + "="*90)
print("FINAL BENCHMARK RESULTS")
print("="*90)
print(df.to_string())
print("="*90)

# Determine best model per metric
best_acc_model = max(all_results, key=lambda n: all_results[n]["final_metrics"]["accuracy"])
best_r2_model = max(all_results, key=lambda n: all_results[n]["final_metrics"]["r2"])
fastest_model = min(all_results, key=lambda n: all_results[n]["train_time"])
smallest_model = min(all_results, key=lambda n: all_results[n]["params"])

print(f"\n  Best Classification Accuracy : {best_acc_model}")
print(f"  Best Regression R²           : {best_r2_model}")
print(f"  Fastest Training             : {fastest_model}")
print(f"  Smallest Model               : {smallest_model}")

## Analysis & Discussion

### Linear Attention Efficiency vs. Accuracy

The **Linear Attention ViT** (XCA-based) achieves O(N·d²) complexity per layer compared to O(N²·d) for standard attention. For this task with N=64 tokens (8×8 patches from a 64×64 image) and d=32 (head dimension), this represents a theoretical speedup when N > d. In practice at 64×64 resolution the two approaches are roughly comparable, but at higher resolutions (e.g., 256×256 with small patches) linear attention would show a significant advantage.

### Hybrid CNN+ViT Benefits

The **Hybrid CNN+ViT** combines the inductive biases of CNNs (translation equivariance, local feature hierarchies) with the global context modeling of transformers. This is particularly beneficial for particle detector images where local energy deposits carry important information. The CNN stem effectively reduces the spatial resolution before the transformer, resulting in fewer tokens and faster attention.

### Recommendation for Particle Physics

- For **production inference** at scale: Linear Attention ViT offers the best efficiency
- For **maximum accuracy**: Hybrid CNN+ViT leverages domain-appropriate local feature extraction  
- For **simplicity and interpretability**: Standard ViT serves as the baseline

### Limitations & Future Work

- Experiments were run on synthetic / small data; real CMS jet images may yield different relative rankings
- Pretraining with masked autoencoders (MAE) could significantly boost all three models
- Architecture search over depth, width, and patch size is recommended
- Consider adding cross-attention between modalities or energy-weighted attention mechanisms

### References

- **ViT**: Dosovitskiy et al., *"An Image is Worth 16×16 Words"*, ICLR 2021
- **XCiT**: El-Nouby et al., *"XCiT: Cross-Covariance Image Transformers"*, NeurIPS 2021  
- **L2ViT**: Zheng, *"The Linear Attention Resurrection in Vision Transformer"*, arXiv:2501.16182, 2025
- **MAE**: He et al., *"Masked Autoencoders Are Scalable Vision Learners"*, CVPR 2022